# From Local Tools to Managed APIs, Hands On

Earlier labs gave the agent its tools with `@tool` decorators, so the tool code lived *inside* the agent.
This notebook takes them out. Horizon Travel's booking data sits behind a Lambda function, and the agent
reaches it through an **AgentCore Gateway** that speaks MCP and discovers the tools automatically — with no
`@tool` code in the agent at all.

It rebuilds `demo.py` one idea at a time, so each piece can be run and watched rather than read. It is a
lesson that happens to arrive at the same code, not a copy of it.

**Part A — no Gateway, no AWS resources.** Steps 1 to 5 run in this kernel. They make no service calls at
all until the very end of Part A, where a Bedrock model call is clearly marked. They create nothing.

**Part B — the real Gateway.** Deploying the Lambda and creating the Gateway are real AWS resources, and
that part is gated: we stop and decide before building anything.

Run the cells in order. Each step ends with output you can check against what the markdown predicted.

## Step 1 — Setup, and where credentials come from

This notebook eventually calls Amazon Bedrock (for the model) and AgentCore Gateway (for the tools). Both
need AWS credentials, and **the notebook never contains them**. boto3 resolves them from the environment,
and on this machine that environment is redirected to `~/.aws-personal/` by two variables.

`demo.py` runs deployed, where AgentCore Runtime supplies credentials. Here in a kernel we point boto3 at the
personal credential store ourselves — as the first thing, before anything constructs a client, because a
boto3 session caches whatever it resolves on first use.

Everything printed by this cell is designed to be safe to commit: notebook outputs are saved inside the
`.ipynb`, so paths are shown relative to the repo root and identifiers are masked. The masking helpers are
defined here and used throughout Part B.

**What to expect:** two package versions, a repo-relative working directory, and a masked identity/account.
The final four account digits should be `6087`, identifying the Vocareum lab account that pays for later
model calls, without publishing the full identifier in notebook output.
`NO CREDENTIALS FOUND` means the kernel was launched without the environment; relaunch it from a shell that
has the two paths set.

Free — the only call here is a read-only `get_caller_identity`.

In [11]:
# =====================================================
# 1. IMPORTS AND CREDENTIAL RESOLUTION
# =====================================================
import json
import logging
import os
from importlib.metadata import version
from pathlib import Path

# Point boto3 at the personal credential store BEFORE any client is built. A boto3
# session caches what it resolves on first use, so this has to come first.
os.environ["AWS_CONFIG_FILE"] = os.path.expanduser("~/.aws-personal/config")
os.environ["AWS_SHARED_CREDENTIALS_FILE"] = os.path.expanduser("~/.aws-personal/credentials")

import boto3
from botocore.exceptions import ClientError, NoCredentialsError

REGION = "us-east-1"


# =====================================================
# 2. LOGGING - for a kernel, not a container
# =====================================================
logging.basicConfig(
    level=logging.WARNING,               # keeps botocore and urllib3 quiet
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    force=True,                          # override any handler the kernel installed
)
logger = logging.getLogger("WanderBot.Gateway")
logger.setLevel(logging.INFO)


# =====================================================
# 3. PRINTING HELPERS - keep saved output publishable
# =====================================================
# Notebook outputs are committed inside the .ipynb, so two things need care:
# absolute paths (they carry the machine's username) and AWS identifiers.


def repo_relative(path):
    """Render a path relative to the repository root.

    Walks up to the directory containing .git, so the home prefix stays out of
    saved output while the useful part of the path remains.

    Args:
        path: A filesystem path, str or Path.

    Returns:
        str: The path relative to the repo root, or just the leaf if it is not
            inside a repository.
    """
    path = Path(path)
    for parent in [path, *path.parents]:
        if (parent / ".git").is_dir():
            return str(path.relative_to(parent))
    return path.name


def mask_id(value, keep_last=4):
    """Mask an identifier, keeping the final characters.

    Enough tail is left to recognise which account or resource this is when
    comparing against the console, not enough to reuse.

    Args:
        value: The identifier to mask.
        keep_last: How many trailing characters to leave visible.

    Returns:
        str: The value with all but its last keep_last characters replaced by
            asterisks.
    """
    s = str(value)
    if len(s) <= keep_last:
        return "*" * len(s)
    return "*" * (len(s) - keep_last) + s[-keep_last:]


def mask_arn(arn):
    """Mask an ARN's account id and unique suffix, keeping its structure.

    The service, region, resource type and human-chosen name stay readable; the
    account id and any generated suffix are hidden.

    Args:
        arn: An ARN string.

    Returns:
        str: The ARN with account and suffix masked, or mask_id(arn) when the
            input is not ARN-shaped.
    """
    parts = str(arn).split(":")
    if len(parts) < 6 or parts[0] != "arn":
        return mask_id(arn)
    parts[4] = mask_id(parts[4])                        # account id
    kind, sep, name = parts[5].partition("/")
    if sep:
        head, dash, tail = name.rpartition("-")
        parts[5] = f"{kind}/{head}{dash}{'*' * len(tail)}" if dash else f"{kind}/{mask_id(name)}"
    return ":".join(parts)


# =====================================================
# 4. PATHS - this notebook's own directory
# =====================================================
BASE_DIR = Path.cwd()
# The Lambda source and schema sit beside this notebook; fall back one level up
# in case the kernel started in the repo root.
CANDIDATES = [BASE_DIR, BASE_DIR / "lab_5" / "Demo"]
DEMO_DIR = next((p for p in CANDIDATES if (p / "booking_lambda.py").is_file()), None)


# =====================================================
# 5. VERIFY - every assumption the rest of the notebook rests on
# =====================================================
print("strands-agents   :", version("strands-agents"))
print("mcp              :", version("mcp"))
print("demo dir         :", repo_relative(DEMO_DIR) if DEMO_DIR else "NOT FOUND")

if DEMO_DIR is None:
    raise FileNotFoundError("booking_lambda.py not found beside this notebook")

try:
    who = boto3.client("sts", region_name=REGION).get_caller_identity()
    print("identity         :", mask_arn(who["Arn"]))
    print("account          :", mask_id(who["Account"]))
    print("\nready for Step 2.")
except NoCredentialsError:
    print("\nNO CREDENTIALS FOUND - boto3 located no usable credential file.")
    print("The cause is this kernel's environment, not the code, and no cell can repair it:")
    print("  1. quit the IDE completely (Cmd-Q, not just closing the window)")
    print("  2. relaunch it from a shell where AWS_CONFIG_FILE / AWS_SHARED_CREDENTIALS_FILE are set")
    print("  3. reopen this notebook and Restart Kernel, then re-run from the top")
except ClientError as exc:
    print("\nCredentials were FOUND but AWS rejected them:", exc.response["Error"]["Code"])
    print("Resolution worked; the credentials themselves need refreshing.")

strands-agents   : 1.53.0
mcp              : 1.29.1
demo dir         : lab_5/Demo
identity         : arn:aws:sts::********6465:assumed-role/***************************8564
account          : ********6465

ready for Step 2.


## Step 2 — Why take the tools out of the agent?

In the earlier labs a tool was a decorated function living in the agent's own source: `@tool` above a
Python function, and Strands turned its signature and docstring into a schema the model could call. That is
the right thing for development — one file, nothing to deploy, easy to change.

It is the wrong thing for production, for reasons that have nothing to do with the model:

- **The tool and the agent share a lifecycle.** A fix to the booking logic means redeploying the agent, and
  every agent that wants that tool carries its own copy of the code.
- **The tool inherits the agent's runtime.** It runs with the agent's Python version, its dependencies, its
  permissions — even when the tool is really a call into a separate system with separate needs.
- **The data lives elsewhere anyway.** Horizon's bookings are not in the agent. They are behind a service,
  and a `@tool` that reaches them is just a client with the agent's deployment schedule bolted on.

The production shape is to run the tool as its own service — here a Lambda function — and put something
between the agent and that service that both sides understand. That something is the **AgentCore Gateway**.
It speaks MCP (Model Context Protocol) to the agent, invokes the Lambda on the other side, and — the part
that does the work — **discovers the available tools and describes them to the agent automatically**, so the
agent carries no tool code at all.

```
WanderBot (Strands agent)
    |  MCP  (tool discovery + tool calls)
AgentCore Gateway
    |  direct Lambda invocation
booking_lambda.py   (get_booking, list_bookings_by_email)
```

The rest of Part A takes this apart from the bottom up, without a Gateway: first the Lambda as a plain
function (Step 3), then the schema that tells the Gateway what the Lambda offers (Step 4), then the MCP
client the agent uses to discover it (Step 5). Part B wires them together for real.

The cell below is a one-off recap of the *old* way, so the contrast is on the page rather than in memory.

In [12]:
# =====================================================
# RECAP - what a tool looked like when it lived in the agent
# =====================================================
# This is the pattern from the earlier labs. It is here only for contrast; nothing
# in lab 5 uses it. Note where the logic lives: right here, in the agent's process.
from strands import tool


@tool
def get_booking_local(booking_ref: str) -> str:
    """Retrieve a booking by reference (illustrative local version).

    Args:
        booking_ref: Booking reference such as BK-1001.

    Returns:
        A short string. Real logic omitted - the point is only where it would run.
    """
    return f"(pretend booking details for {booking_ref})"


# The decorator wraps the function into a Strands tool. The schema the model sees
# is built from the signature and docstring above - all of it inside the agent.
print("tool name        :", get_booking_local.tool_name)
print("schema keys      :", sorted(get_booking_local.tool_spec.keys()))
print("input properties :", list(get_booking_local.tool_spec["inputSchema"]["json"]["properties"]))
print()
print("The name, the description and the parameter schema all come from code in")
print("this process. Step 3 onward moves every bit of that out to a Lambda.")

tool name        : get_booking_local
schema keys      : ['description', 'inputSchema', 'name']
input properties : ['booking_ref']

The name, the description and the parameter schema all come from code in
this process. Step 3 onward moves every bit of that out to a Lambda.


## Step 3 — The Lambda is just a function

Before any Gateway exists, look at what sits behind it. `booking_lambda.py` is an ordinary Python module
with two functions over some hard-coded data — three sample bookings — and a handler. There is no HTTP
routing, no web framework, nothing AWS-specific in the functions themselves. That is the point the whole
lesson turns on: the tool is a plain function, and everything else is transport.

This step imports that real file and runs it *in this kernel*, with no AWS involved at all. If the logic is
sound here, then anything that goes wrong in Part B is the Gateway or the deployment — not the tool.

### 3a. Call the functions directly

`get_booking` looks up one booking by reference; `list_bookings_by_email` finds all bookings for an address.
Both return JSON strings. The cell imports the module from the demo directory and calls them the way any
Python code would.

**What to expect:** Alice's BK-1001 returned as JSON, both of Alice's bookings for her email, and a plain
not-found message for a reference that does not exist. No credentials are used.

Free — pure local calls.

In [13]:
# =====================================================
# 3a. THE LAMBDA'S FUNCTIONS, CALLED IN-PROCESS
# =====================================================
import sys

# The Lambda file sits in the demo directory; make it importable.
if str(DEMO_DIR) not in sys.path:
    sys.path.insert(0, str(DEMO_DIR))

import booking_lambda

# A single lookup by reference.
print("get_booking('BK-1001'):")
print("  ", booking_lambda.get_booking("BK-1001"))

# The handler upper-cases the reference, so lookup is case-insensitive.
print()
print("get_booking('bk-1001')  (lower case still resolves):")
print("  ", booking_lambda.get_booking("bk-1001")[:60], "...")

# A reference that does not exist - a message, not an exception.
print()
print("get_booking('BK-9999'):")
print("  ", booking_lambda.get_booking("BK-9999"))

# All bookings for one customer email.
print()
print("list_bookings_by_email('alice@example.com'):")
print("  ", booking_lambda.list_bookings_by_email("alice@example.com")[:90], "...")

print()
print("Plain functions over local data. Nothing here knows about a Gateway.")

get_booking('BK-1001'):
   {"booking_ref": "BK-1001", "customer": "Alice Johnson", "email": "alice@example.com", "flight": "HZ-101", "route": "LHR \u2192 CDG", "date": "2026-03-15", "cabin": "Economy Flex", "fare_usd": 189.99, "status": "CONFIRMED"}

get_booking('bk-1001')  (lower case still resolves):
   {"booking_ref": "BK-1001", "customer": "Alice Johnson", "ema ...

get_booking('BK-9999'):
   No booking found with reference BK-9999.

list_bookings_by_email('alice@example.com'):
   {"email": "alice@example.com", "bookings": [{"booking_ref": "BK-1001", "customer": "Alice  ...

Plain functions over local data. Nothing here knows about a Gateway.


### 3b. How the handler knows which function to call

The Gateway does not call `get_booking` directly. It invokes the *Lambda*, once, and tells the handler which
tool was meant. The handler reads that from the invocation's client context:

``` Python
raw_tool = context.client_context.custom.get("bedrockAgentCoreToolName", "")
tool = raw_tool.split("___", 1)[-1] if "___" in raw_tool else raw_tool
```

The Gateway passes a **prefixed** name like `booking-target___get_booking` — target name, three underscores,
tool name — and the handler keeps the part after the separator. That is the routing: one Lambda, several
tools, dispatched by a name the Gateway sets.

To exercise this without a Gateway, the cell builds a small fake `context` object with just the attributes
the handler reads, then calls the handler as the Gateway would. It also shows the fallback path — when there
is no client context at all, the handler infers the tool from the event's keys, which is what makes a plain
`aws lambda invoke` still work.

**What to expect:** the prefixed name routes to `get_booking` and returns a `200` with the booking in the
body; the fallback routes `{"email": ...}` to `list_bookings_by_email`. The handler prints its own
`Tool called: ...` line each time.

Free — still no AWS.

In [14]:
# =====================================================
# 3b. THE HANDLER'S TOOL-NAME ROUTING, VIA A FAKE CONTEXT
# =====================================================


class FakeClientContext:
    """The minimal client_context the handler reads from.

    A real Lambda context carries far more; the handler only touches
    client_context.custom.get("bedrockAgentCoreToolName"), so that is all this
    reproduces.
    """

    def __init__(self, tool_name):
        """Store the tool name under the key the Gateway uses.

        Args:
            tool_name: The value the Gateway would place in
                bedrockAgentCoreToolName, e.g. "booking-target___get_booking".
        """
        # .custom must be an object with a .get method; a dict provides exactly that.
        self.custom = {"bedrockAgentCoreToolName": tool_name}


class FakeContext:
    """Stand-in for the Lambda context object, carrying only client_context."""

    def __init__(self, tool_name):
        """Wrap a FakeClientContext.

        Args:
            tool_name: Passed through to FakeClientContext.
        """
        self.client_context = FakeClientContext(tool_name)


# ---- the Gateway path: a prefixed tool name in the client context ----
gateway_context = FakeContext("booking-target___get_booking")
result = booking_lambda.lambda_handler({"booking_ref": "BK-1001"}, gateway_context)

print("invoked as the Gateway would, tool = 'booking-target___get_booking'")
print("  statusCode:", result["statusCode"])
print("  body      :", result["body"][:80], "...")

# ---- the fallback path: no client context, tool inferred from event keys ----
print()
fallback = booking_lambda.lambda_handler({"email": "bob@example.com"}, None)
print("invoked with context=None (e.g. plain 'aws lambda invoke')")
print("  the handler saw an 'email' key and routed to list_bookings_by_email")
print("  body      :", fallback["body"][:80], "...")

Tool called: get_booking | Event: {"booking_ref": "BK-1001"}
invoked as the Gateway would, tool = 'booking-target___get_booking'
  statusCode: 200
  body      : {"result": "{\"booking_ref\": \"BK-1001\", \"customer\": \"Alice Johnson\", \"em ...

Tool called: list_bookings_by_email | Event: {"email": "bob@example.com"}
invoked with context=None (e.g. plain 'aws lambda invoke')
  the handler saw an 'email' key and routed to list_bookings_by_email
  body      : {"result": "{\"email\": \"bob@example.com\", \"bookings\": [{\"booking_ref\": \" ...


## Step 4 — The schema is the bridge

Step 3 showed the Lambda is just functions. But a function's name, its description and its parameters are
Python — the Gateway cannot see them, and neither can the model. Something has to *describe* the tools in a
form both understand. That something is the schema in `booking_lambda.json`.

You register it when you add the Lambda as a target to the Gateway. It is a list of tool definitions, one
per function the Lambda exposes, and each definition has three parts:

- **`name`** — the tool the model calls, and the suffix the Gateway puts after `___` in the client context
  (Step 3b). `get_booking` here is why the handler routed on `get_booking` there.
- **`description`** — prose the model reads when deciding whether this tool fits the request.
- **`inputSchema`** — a JSON Schema for the parameters, so the model knows what to send and the Gateway can
  validate it.

Here is the thing worth noticing. This is the *same* information Step 2's `@tool` produced — name,
description, parameter schema — and the *same* three-part shape. The decorator generated it from the
function's signature and docstring automatically. For a Gateway tool you write it by hand instead, because
the Gateway sits in a different process from the function and cannot introspect it. Same contract, authored
in a different place.

The cell reads the real schema file and lines its shape up against the tool spec from Step 2.

Free — reading a local file.

In [15]:
# =====================================================
# 4. THE TOOL SCHEMA THE GATEWAY REGISTERS
# =====================================================

schema_path = DEMO_DIR / "booking_lambda.json"
schema = json.loads(schema_path.read_text())

print("tools defined in", repo_relative(schema_path), ":", len(schema))
print()

for tool_def in schema:
    required = tool_def["inputSchema"].get("required", [])
    params = list(tool_def["inputSchema"]["properties"])
    print("  name        :", tool_def["name"])
    print("  description :", tool_def["description"][:64], "...")
    print("  parameters  :", params, " required:", required)
    print()

# The same three keys the decorator produced in Step 2, authored by hand here.
gateway_keys = sorted(schema[0].keys())
local_keys = sorted(get_booking_local.tool_spec.keys())
print("Gateway schema keys per tool :", gateway_keys)
print("Step 2 @tool tool_spec keys  :", local_keys)
print("same three-part contract     :", gateway_keys == local_keys)

tools defined in lab_5/Demo/booking_lambda.json : 2

  name        : get_booking
  description : Retrieve full details for a single booking using its booking ref ...
  parameters  : ['booking_ref']  required: ['booking_ref']

  name        : list_bookings_by_email
  description : List all bookings associated with a customer email address. ...
  parameters  : ['email']  required: ['email']

Gateway schema keys per tool : ['description', 'inputSchema', 'name']
Step 2 @tool tool_spec keys  : ['description', 'inputSchema', 'name']
same three-part contract     : True


## Step 5 — What does the agent actually connect to?

The architecture now has both ends. Step 3 proved the Lambda logic and routing; Step 4 supplied the contract
that describes its tools. What remains is the agent-facing half: open an MCP session to the Gateway, ask the
server which tools exist, and hand those tools to Strands.

Three layers are easy to blur together:

| Layer | Object in the demo | Responsibility |
| --- | --- | --- |
| **Transport** | `streamable_http_client(...)` | moves MCP messages over HTTP to the Gateway endpoint |
| **MCP session** | `MCPClient(...)` | owns connection lifetime, performs the MCP handshake, lists tools, calls tools |
| **Agent** | `Agent(tools=tools)` | gives schemas to the model and invokes whichever tool the model selects |

The Gateway is the **MCP server**. `MCPClient` is exactly what its name says: the client. Lambda never speaks
MCP itself — the Gateway translates the MCP tool call into a direct Lambda invocation.

Everything in this step is local. Without a Gateway endpoint we cannot prove what tools the server returns,
but we can prove when networking starts, what fails before a session exists, and why discovered tools fit
straight into `Agent(tools=...)`.

### 5a. Construct the client without connecting

The demo passes a function into `MCPClient`, not an open connection:

``` Python
MCPClient(lambda: streamable_http_client(url=GATEWAY_ENDPOINT))
```

That `lambda` is a **transport factory** — a recipe the client can call later. Construction stores the recipe;
it does not execute it. Entering `with client:` is the boundary where real work starts: `__enter__` calls
`start()`, starts a background thread, opens the transport, performs MCP initialisation, and blocks until the
session is ready or the startup timeout expires.

The cell makes that visible with a named factory and a counter. The endpoint uses `.invalid`, a reserved domain,
but no request should reach it because the context is never entered. It then calls `list_tools_sync()` too
early on purpose; the method should refuse locally with `MCPClientInitializationError` rather than attempting
network access.

**What to expect:** factory call count remains zero before and after the failed listing, and the error says the
client session is not running.

Free — no network call.

In [16]:
# =====================================================
# 5a. CONSTRUCTION IS LAZY; THE WITH BLOCK OPENS THE SESSION
# =====================================================
from mcp.client.streamable_http import streamable_http_client
from strands.tools.mcp.mcp_client import MCPClient, MCPClientInitializationError

PLACEHOLDER_ENDPOINT = "https://example.invalid/mcp"
factory_calls = 0


def make_transport():
    """Build the streamable-HTTP transport when MCPClient asks for it.

    This function is the expanded, readable form of the demo's lambda. Merely
    passing it to MCPClient does not run it; entering `with client:` does.

    Returns:
        An asynchronous context manager that opens MCP streams to the placeholder
        endpoint when entered.
    """
    global factory_calls
    factory_calls = factory_calls + 1
    return streamable_http_client(url=PLACEHOLDER_ENDPOINT)


client = MCPClient(make_transport)

print("transport factory calls after MCPClient(...):", factory_calls)
print("supports 'with client:'                    :", hasattr(client, "__enter__"))
print("supports list_tools_sync()                 :", hasattr(client, "list_tools_sync"))

print()
print("calling list_tools_sync() before entering the context:")
try:
    client.list_tools_sync()
    print("  succeeded - unexpected")
except MCPClientInitializationError as error:
    print(" ", type(error).__name__ + ":", error)

print()
print("transport factory calls after that failure :", factory_calls)
print("Zero means the failure happened locally; no request reached example.invalid.")

transport factory calls after MCPClient(...): 0
supports 'with client:'                    : True
supports list_tools_sync()                 : True

calling list_tools_sync() before entering the context:
  MCPClientInitializationError: the client session is not running. Ensure the agent is used within the MCP client context manager. For more information see: https://strandsagents.com/docs/user-guide/concepts/tools/mcp-tools/#mcpclientinitializationerror-python

transport factory calls after that failure : 0
Zero means the failure happened locally; no request reached example.invalid.


### 5b. Why can discovered tools go straight into `Agent`?

`list_tools_sync()` receives MCP tool definitions from the server and wraps each one as an `MCPAgentTool`.
That class and Step 2's decorated local tool look unrelated — one calls a remote MCP server, the other calls a
Python function — but both implement Strands' `AgentTool` interface.

That common interface is the whole reason the agent code barely changes. `Agent` asks every tool for a
`tool_name`, a `tool_spec`, and a way to invoke it; it does not care which subclass supplies them. The model
sees the same name, description and input schema either way and cannot infer where execution happens.

The cell prints the inheritance chains for both types and checks the shared base class.

Free — type inspection only.

In [17]:
# =====================================================
# 5b. LOCAL AND REMOTE TOOLS CONVERGE ON AgentTool
# =====================================================
import inspect

from strands.tools.mcp.mcp_agent_tool import MCPAgentTool
from strands.types.tools import AgentTool


def inheritance_chain(value_or_class):
    """Return readable class names from a value's inheritance hierarchy.

    Args:
        value_or_class: Either a class or an instance whose class should be
            inspected.

    Returns:
        list[str]: Class names from most specific to most general.
    """
    if inspect.isclass(value_or_class):
        class_to_inspect = value_or_class
    else:
        class_to_inspect = type(value_or_class)

    names = []
    for base in inspect.getmro(class_to_inspect):
        names.append(base.__name__)
    return names


print("Step 2 local tool:")
print("  chain:", " -> ".join(inheritance_chain(get_booking_local)))
print("  is AgentTool:", isinstance(get_booking_local, AgentTool))

print()
print("Gateway-discovered tool class:")
print("  chain:", " -> ".join(inheritance_chain(MCPAgentTool)))
print("  is AgentTool:", issubclass(MCPAgentTool, AgentTool))

print()
print("Both satisfy the interface Agent(tools=...) consumes.")
print("Different execution location; the same contract from the agent's perspective.")

Step 2 local tool:
  chain: DecoratedFunctionTool -> AgentTool -> ABC -> Generic -> object
  is AgentTool: True

Gateway-discovered tool class:
  chain: MCPAgentTool -> AgentTool -> ABC -> object
  is AgentTool: True

Both satisfy the interface Agent(tools=...) consumes.
Different execution location; the same contract from the agent's perspective.


### 5c. Put the connection, discovery and agent call in the right lifetime

The exact production flow belongs in one `with` block:

1. Construct `MCPClient` with a transport factory.
2. Enter the context — open the HTTP transport and initialise the MCP session.
3. Call `list_tools_sync()` — ask the Gateway's MCP server for every registered tool and adapt each to
   `MCPAgentTool`.
4. Build the Strands agent with that list, then invoke it **while the client remains open**.
5. Leave the context — close the session and background thread, even if an exception occurred.

That fourth point is load-bearing. Do not return `tools` from the context and invoke them later: an
`MCPAgentTool` calls back through its `MCPClient`, and that client has already been stopped after `__exit__`.
Discovery and use share one session lifetime.

The function below is the demo's entrypoint logic in a reusable shape. It is defined but not called because the
Gateway endpoint does not exist yet. Notice there is no `@tool` anywhere.

> **Note:** `mcp` 1.29.1 exports both `streamable_http_client` and `streamablehttp_client`, and they are
> different functions with different parameter surfaces — not spelling aliases. This course's `demo.py` uses
> `streamable_http_client`, so the notebook uses that exact function rather than substituting the similar name.

Free — definition only; no client is entered.

In [18]:
# =====================================================
# 5c. THE COMPLETE AGENT-SIDE FLOW, DEFINED BUT NOT CALLED
# =====================================================
from strands import Agent


GATEWAY_SYSTEM_PROMPT = """You are WanderBot, the AI travel assistant for Horizon Travel.

Use the Gateway tools to retrieve bookings by reference or customer email.
Present results clearly, and never invent booking details."""


def invoke_through_gateway(endpoint, user_message, model):
    """Discover Gateway tools and answer one user message while connected.

    The client, discovered tools and agent invocation all remain inside one
    context-managed MCP session. Leaving the block closes the transport and its
    background thread automatically.

    Args:
        endpoint: The Gateway MCP URL, ending in /mcp.
        user_message: The customer's question.
        model: A configured Strands model provider, such as BedrockModel.

    Returns:
        AgentResult: The final response from the Strands agent.
    """
    # `lambda: expr` is a no-argument function whose body is `expr` - "compute
    # this, but later". Passing it (not the result) means the transport is NOT
    # created here: MCPClient stores the function and calls it inside the `with`
    # block, on its background thread. Same as defining a one-line
    # make_transport() and passing make_transport.
    gateway_client = MCPClient(
        lambda: streamable_http_client(url=endpoint)
    )

    with gateway_client:
        tools = gateway_client.list_tools_sync()
        print("discovered tools:", [tool.tool_name for tool in tools])

        agent = Agent(
            model=model,
            system_prompt=GATEWAY_SYSTEM_PROMPT,
            tools=tools,
        )

        # Invoke before leaving the block. Remote tools need this open client.
        response = agent(user_message)

    return response


print("invoke_through_gateway defined, not called.")
print("No endpoint was opened and no tools were discovered yet.")

invoke_through_gateway defined, not called.
No endpoint was opened and no tools were discovered yet.


### What the output proved

**Construction is not connection.** `MCPClient(make_transport)` left the factory count at zero. Calling
`list_tools_sync()` before `with client:` raised `MCPClientInitializationError: the client session is not
running`, and the count stayed zero afterwards — the refusal was local, with no request to the placeholder.

**The context manager is the session boundary.** From the installed source, `__enter__` calls `start()`, which
starts a background thread and waits for MCP initialisation; `__exit__` calls `stop()`, which closes the session,
event loop and thread. That is why discovery *and tool use* belong inside the same block.

**Local and remote tools meet at the same interface.** The local `DecoratedFunctionTool` and remote
`MCPAgentTool` both inherit `AgentTool`. So this line needs no special Gateway mode:

``` Python
agent = Agent(tools=tools)
```

The model receives names, descriptions and JSON schemas in either case. It does not know whether executing one
runs a local function, crosses an MCP connection to a Gateway, or ends in a Lambda. That location is an
implementation detail below the `AgentTool` boundary.

What Part A cannot prove is equally important: there is no server yet, so `list_tools_sync()` has not returned
a real tool, no MCP handshake has completed, and no Gateway has invoked the Lambda. Those are Part B.

# Part A checkpoint — what is proven before creating anything

| Step | Question | Answer established locally |
| --- | --- | --- |
| 1 | why move tools out? | local tools couple logic, dependencies, permissions and deployment to the agent |
| 2 | what is behind the Gateway? | ordinary Lambda functions plus a handler that routes the Gateway's tool name |
| 3 | what does the schema contribute? | the same name/description/input-schema contract `@tool` generated locally |
| 4 | how does the agent consume remote tools? | an MCP session discovers `MCPAgentTool` objects implementing `AgentTool` |

The architecture is now explained from both ends without deploying a Lambda or creating a Gateway:

```
Agent(tools=[MCPAgentTool, ...])
    |  MCP over streamable HTTP  - owned by MCPClient's with block
AgentCore Gateway
    |  validates arguments from booking_lambda.json
    |  sets bedrockAgentCoreToolName in Lambda client context
Lambda handler
    |  strips target___ prefix and routes by tool name
get_booking / list_bookings_by_email
```

## Part B gate

Part B creates real resources and proves the missing links: deploy the Lambda, create a Gateway with a Lambda
target and this schema, copy its `/mcp` endpoint, run `list_tools_sync()`, and make a real booking lookup.

The course uses the AgentCore console and **No Authentication** for development. An unauthenticated Gateway URL
is a capability: anyone holding it can invoke the exposed tools, so it must not be hard-coded into this public
notebook or saved in output. Part B will request it interactively for the kernel's lifetime and will mask it
when printed.

Stop here until those resources are deliberately created.

# Part B — the real Gateway, as one idempotent stack

Part A explained both ends without creating anything. Part B creates the middle, and does it differently
from the course. The course clicks through the console: create a Gateway, add a Lambda target, paste the
schema, copy the URL into the agent's source. That works once. It does not survive a second run, it leaves no
record of what was created, and the URL ends up hard-coded in a file.

Here the whole backend is **one CloudFormation stack**, deployed from this notebook with boto3:

```
wanderbot-gateway-demo  (CloudFormation stack)
  |- LambdaRole        IAM role the function runs as
  |- BookingLambda     booking_lambda.py, inlined byte-for-byte from Step 3
  |- GatewayRole       IAM role the Gateway assumes to invoke the Lambda
  |- Gateway           AWS::BedrockAgentCore::Gateway, MCP, no authentication
  |- BookingTarget     AWS::BedrockAgentCore::GatewayTarget, schema from Step 4
  outputs: GatewayUrl, LambdaArn
```

Three things this buys that the console does not:

- **Idempotent by construction.** A stack has a name. Run the deploy again and CloudFormation computes the
  difference and applies only that — or reports there is nothing to do. No orphaned second Gateway.
- **The endpoint is an output, not a paste.** `GatewayUrl` is a read-only property of the Gateway resource,
  so the stack hands the `/mcp` URL back and the notebook reads it. Nothing is copied by hand or committed.
- **Teardown is one call.** `delete_stack` removes all five resources in dependency order.

CloudFormation has native `AWS::BedrockAgentCore::Gateway` and `GatewayTarget` types, confirmed by listing the
registered types in this region before writing a line.

**A note on accounts.** This notebook was first run against a lab account whose cost-control policy
(`voc-cancel-cred`) intermittently denied CloudFormation stack operations. The backend was built there by hand
instead — five direct API calls — and that version survives as Step 6d, a fallback that works in any account.
Part B now runs against an account where the stack is permitted, which is why the stack is the primary path.

**This part costs money and creates resources.** Each step says what it creates. Step 9 deletes everything.

## Step 6 — Build the stack

### 6a. The template, as a Python dictionary

A CloudFormation template is JSON, so building it as a `dict` and letting `json.dumps` serialise it avoids
every quoting problem of hand-written YAML and lets the notebook read the two source files directly. Two
things are loaded from disk rather than typed in:

- The Lambda code comes from `booking_lambda.py` through the `ZipFile` property, which accepts inline source
  up to 4,096 bytes. The file is 3,850 — so the function that deploys is exactly the one Step 3 tested.
- The tool schema comes from `booking_lambda.json` straight into the target's `InlinePayload`. Step 4's
  three-part contract is now literally the registration.

Two IAM roles, and it matters which is which. `LambdaRole` is what the *function* runs as — basic logging
only. `GatewayRole` is what the *Gateway* assumes when it invokes the function, so it trusts
`bedrock-agentcore.amazonaws.com` and is allowed `lambda:InvokeFunction` on this one ARN. The course lets the
console create that second role invisibly; here it is on the page.

`AuthorizerType: NONE` is the course's "No Authentication". Step 1 of Part A said why that URL is then a
capability, which is also why `GatewayUrl` is masked wherever it is printed.

**The casing trap, which this notebook fell into on its first deploy.** CloudFormation's property names are
PascalCase all the way down — `Mcp` / `Lambda` / `ToolSchema` / `InlinePayload`, and inside each tool `Name` /
`Description` / `InputSchema`. The control-plane API that 6d's fallback calls uses lowerCamel for the same
structure, and `booking_lambda.json` is written in *that* form. Load the file into the template verbatim and the
target fails validation with "required key [Name] not found; extraneous key [name] is not permitted" — and
the whole stack rolls back. Fix those three keys and it fails again, one level deeper: `InputSchema: required
key [Type] not found`. CloudFormation does not treat the input schema as opaque JSON Schema; it models it as its
own typed `SchemaDefinition`, PascalCase and *recursive* — every property's schema, and `Items`, is another
`SchemaDefinition`. Two more rules ride along: the `Type` *value* stays lowercase (it is an enum of the six
JSON Schema types), and property *names* like `booking_ref` are user data and are not touched.
`to_cfn_schema()` walks the tree applying exactly that. Same contract, two spellings, chosen by which API
consumes it — and it took two failed deploys to find the bottom of it.

**A third deploy failed differently, and the difference is the point.** The first two were CloudFormation
rejecting the *template* before touching AWS. The third passed validation, the handler called the Gateway
service, and the *service* refused: `CredentialProviderConfigurations is required for Lambda targets`. The CFN
type schema does not list that property as required — so `validate_template` and property validation both let
it through, and only the real API caught it. It names how the Gateway authenticates to the target; for a Lambda
that is the Gateway's own role, `GATEWAY_IAM_ROLE`. The console sets it silently, which is why a console-built
Gateway never shows you this field exists.

`build_template()` returns the dict. Nothing is sent — the cell ends by printing a summary.

Free — builds a dictionary.

In [19]:
# =====================================================
# 6a. THE CLOUDFORMATION TEMPLATE, BUILT AS A DICT
# =====================================================

STACK_NAME = "wanderbot-gateway-demo"
LAMBDA_FUNCTION_NAME = "wanderbot-booking-tools"
GATEWAY_NAME = "wanderbot-gateway"
TARGET_NAME = "booking-target"      # becomes the prefix before ___ in tool names
LAMBDA_RUNTIME = "python3.12"       # matches the Python that ran Step 3 locally


def to_cfn_schema(json_schema):
    """Convert a JSON Schema node to CloudFormation's typed SchemaDefinition, recursively.

    CloudFormation does NOT accept raw JSON Schema here. It models InputSchema as
    its own structure with PascalCase keys - Type / Properties / Required / Items
    / Description - and that structure is recursive: every value in Properties,
    and Items itself, is another SchemaDefinition. Three rules, each learned from
    a failed deploy or read from the type schema:

      * KEYS become PascalCase            ("type" -> "Type")
      * the Type VALUE stays lowercase    ("string" stays "string" - it is an enum)
      * property NAMES are user data       ("booking_ref" is left exactly as is)

    Args:
        json_schema: A JSON Schema node as written in booking_lambda.json.

    Returns:
        dict: The equivalent CloudFormation SchemaDefinition.
    """
    converted = {"Type": json_schema["type"]}

    if "description" in json_schema:
        converted["Description"] = json_schema["description"]

    if "required" in json_schema:
        converted["Required"] = json_schema["required"]

    if "properties" in json_schema:
        # Recurse into each property's own schema; keep the property name verbatim.
        converted["Properties"] = {
            property_name: to_cfn_schema(property_schema)
            for property_name, property_schema in json_schema["properties"].items()
        }

    if "items" in json_schema:
        converted["Items"] = to_cfn_schema(json_schema["items"])

    return converted


def to_cfn_tool(tool_definition):
    """Re-case one tool definition from the control-plane API's form to CloudFormation's.

    booking_lambda.json is written for the bedrock-agentcore-control API, which
    uses lowerCamel keys. CloudFormation wants the same contract in PascalCase,
    all the way down - the tool's three fields AND the schema tree inside
    InputSchema, which to_cfn_schema handles.

    Args:
        tool_definition: One entry from the schema file.

    Returns:
        dict: The same tool in CloudFormation's form.
    """
    return {
        "Name": tool_definition["name"],
        "Description": tool_definition["description"],
        "InputSchema": to_cfn_schema(tool_definition["inputSchema"]),
    }


def build_template(lambda_source, tool_schema):
    """Build the CloudFormation template for the whole Gateway backend.

    Args:
        lambda_source: The full text of booking_lambda.py, inlined as the
            function's code. Must be under 4,096 bytes for ZipFile.
        tool_schema: The parsed booking_lambda.json - a list of tool
            definitions, each with name / description / inputSchema.

    Returns:
        dict: A CloudFormation template declaring two IAM roles, the Lambda,
            the Gateway and its Lambda target, with GatewayUrl and LambdaArn as
            outputs.
    """
    return {
        "AWSTemplateFormatVersion": "2010-09-09",
        "Description": "WanderBot lab 5: Lambda tools fronted by an AgentCore Gateway (MCP, no auth)",
        "Resources": {

            # ---- the role the FUNCTION runs as: logging only ----
            "LambdaRole": {
                "Type": "AWS::IAM::Role",
                "Properties": {
                    "AssumeRolePolicyDocument": {
                        "Version": "2012-10-17",
                        "Statement": [{
                            "Effect": "Allow",
                            "Principal": {"Service": "lambda.amazonaws.com"},
                            "Action": "sts:AssumeRole",
                        }],
                    },
                    "ManagedPolicyArns": [
                        "arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
                    ],
                },
            },

            # ---- the function, source inlined from booking_lambda.py ----
            "BookingLambda": {
                "Type": "AWS::Lambda::Function",
                "Properties": {
                    "FunctionName": LAMBDA_FUNCTION_NAME,
                    "Runtime": LAMBDA_RUNTIME,
                    "Handler": "index.lambda_handler",   # ZipFile code is saved as index.py
                    "Role": {"Fn::GetAtt": ["LambdaRole", "Arn"]},
                    "Timeout": 10,
                    "Code": {"ZipFile": lambda_source},
                },
            },

            # ---- the role the GATEWAY assumes to invoke the function ----
            "GatewayRole": {
                "Type": "AWS::IAM::Role",
                "Properties": {
                    "AssumeRolePolicyDocument": {
                        "Version": "2012-10-17",
                        "Statement": [{
                            "Effect": "Allow",
                            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                            "Action": "sts:AssumeRole",
                        }],
                    },
                    "Policies": [{
                        "PolicyName": "InvokeBookingLambda",
                        "PolicyDocument": {
                            "Version": "2012-10-17",
                            "Statement": [{
                                "Effect": "Allow",
                                "Action": "lambda:InvokeFunction",
                                "Resource": {"Fn::GetAtt": ["BookingLambda", "Arn"]},
                            }],
                        },
                    }],
                },
            },

            # ---- the Gateway: an MCP server with no authentication ----
            "Gateway": {
                "Type": "AWS::BedrockAgentCore::Gateway",
                "Properties": {
                    "Name": GATEWAY_NAME,
                    "Description": "Fronts the WanderBot booking Lambda over MCP",
                    "ProtocolType": "MCP",
                    "AuthorizerType": "NONE",
                    "RoleArn": {"Fn::GetAtt": ["GatewayRole", "Arn"]},
                },
            },

            # ---- the target: this Lambda, described by Step 4's schema ----
            "BookingTarget": {
                "Type": "AWS::BedrockAgentCore::GatewayTarget",
                "Properties": {
                    "GatewayIdentifier": {"Fn::GetAtt": ["Gateway", "GatewayIdentifier"]},
                    "Name": TARGET_NAME,
                    "Description": "Booking lookup tools",
                    "TargetConfiguration": {
                        "Mcp": {
                            "Lambda": {
                                "LambdaArn": {"Fn::GetAtt": ["BookingLambda", "Arn"]},
                                # Re-cased per tool: the file is lowerCamel, CFN wants PascalCase.
                                "ToolSchema": {"InlinePayload": [to_cfn_tool(t) for t in tool_schema]},
                            }
                        }
                    },
                    # How the Gateway authenticates TO the target. For a Lambda that is
                    # the Gateway's own role (GatewayRole above). The CFN type schema
                    # marks this optional, but the service rejects a Lambda target
                    # without it - the console sets it silently.
                    "CredentialProviderConfigurations": [
                        {"CredentialProviderType": "GATEWAY_IAM_ROLE"}
                    ],
                },
            },
        },

        # ---- what the notebook reads back after deployment ----
        "Outputs": {
            "GatewayUrl": {"Value": {"Fn::GetAtt": ["Gateway", "GatewayUrl"]}},
            "LambdaArn": {"Value": {"Fn::GetAtt": ["BookingLambda", "Arn"]}},
        },
    }


lambda_source = (DEMO_DIR / "booking_lambda.py").read_text()
tool_schema = json.loads((DEMO_DIR / "booking_lambda.json").read_text())

template = build_template(lambda_source, tool_schema)
template_body = json.dumps(template)

print("Lambda source bytes :", len(lambda_source.encode()), "(ZipFile limit 4096)")
print("tools in schema     :", [t["name"] for t in tool_schema])
print("re-cased for CFN    :", json.dumps(to_cfn_tool(tool_schema[0])["InputSchema"]))
print("resources declared  :", list(template["Resources"]))
print("outputs declared    :", list(template["Outputs"]))
print("template size       :", len(template_body), "bytes")
print()
print("Built, not deployed. Nothing has been sent to AWS.")

Lambda source bytes : 3850 (ZipFile limit 4096)
tools in schema     : ['get_booking', 'list_bookings_by_email']
re-cased for CFN    : {"Type": "object", "Required": ["booking_ref"], "Properties": {"booking_ref": {"Type": "string", "Description": "Booking reference number (e.g. BK-1001)"}}}
resources declared  : ['LambdaRole', 'BookingLambda', 'GatewayRole', 'Gateway', 'BookingTarget']
outputs declared    : ['GatewayUrl', 'LambdaArn']
template size       : 6935 bytes

Built, not deployed. Nothing has been sent to AWS.


### 6a′. Validate the shape before sending it — and know what that cannot catch

Three deploys of this template failed, each in a slightly different place, and each cost a two-to-three
minute create-and-rollback round trip to learn one fact. That is slow feedback, and the fix is to move the
check *before* the deploy.

`validate_template` does not do it. It checks template syntax and intrinsic functions, and reports which
capabilities you need — and that is nearly all. A template passes it with every property name wrong.

What does do it: CloudFormation publishes a JSON Schema for every resource type, retrievable with
`describe_type`. Each resource's `Properties` block can be checked against that schema locally with the
`jsonschema` package, in milliseconds, producing the *same* messages the deploy would — "required key [Type]
not found", "extraneous key [type] is not permitted". The function below does that for every resource in the
template. `GetAtt` and other intrinsics are replaced with a placeholder string first, since they are not
values yet; only the shape is being checked.

**And what it cannot catch.** The third failure was `CredentialProviderConfigurations is required for Lambda
targets` — a rule the *service* enforces but the CFN type schema marks optional. No offline check against that
schema can see it. When a CloudFormation type wraps a newer API, cross-check CFN's `required` list against the
API's own: the botocore model for `CreateGatewayTarget` lists that field, and the hand-built fallback in 6d
always passed it. The gap between the two lists is where this class of failure lives.

**What to expect:** zero errors for every resource, or a short list naming the exact path and problem. The two
placeholder mismatches on `GatewayIdentifier` and `LambdaArn` patterns are expected — they are the stand-ins
for intrinsics, not real values.

Free — one `describe_type` read per resource type.

In [20]:
# =====================================================
# 6a'. OFFLINE PROPERTY VALIDATION AGAINST THE REAL TYPE SCHEMA
# =====================================================
import jsonschema

cfn = boto3.client("cloudformation", region_name=REGION)


def replace_intrinsics(value):
    """Swap CloudFormation intrinsics for a placeholder so only shape is checked.

    {"Fn::GetAtt": [...]} and friends are not values until deploy time, and a
    JSON Schema validator would reject them as the wrong type. Replacing each
    with a string lets everything else be validated.

    Args:
        value: Any node of a template's Properties block.

    Returns:
        The same structure with every single-key Fn::* dict replaced by the
        string "intrinsic-placeholder".
    """
    if isinstance(value, dict):
        if len(value) == 1 and next(iter(value)).startswith("Fn::"):
            return "intrinsic-placeholder"
        return {key: replace_intrinsics(inner) for key, inner in value.items()}
    if isinstance(value, list):
        return [replace_intrinsics(inner) for inner in value]
    return value


def validate_resource_properties(template):
    """Check every resource's Properties against its CloudFormation type schema.

    Catches property-shape mistakes - wrong casing, unknown keys, missing
    required keys - before any deploy, with the same messages the deploy would
    produce. Cannot catch rules the service enforces beyond its published
    schema; see the markdown above.

    Args:
        template: The template dict from build_template().

    Returns:
        dict: {logical_id: [error messages]} - empty lists mean clean.
    """
    results = {}
    schema_cache = {}

    for logical_id, resource in template["Resources"].items():
        type_name = resource["Type"]
        if type_name not in schema_cache:
            described = cfn.describe_type(Type="RESOURCE", TypeName=type_name)
            schema_cache[type_name] = json.loads(described["Schema"])

        validator = jsonschema.Draft7Validator(schema_cache[type_name])
        properties = replace_intrinsics(resource.get("Properties", {}))

        messages = []
        for error in validator.iter_errors(properties):
            path = "/".join(str(part) for part in error.path) or "(root)"
            # Placeholder strings fail pattern checks on id/ARN fields; that is
            # expected and not a template problem.
            if "intrinsic-placeholder" in error.message:
                continue
            messages.append(path + ": " + error.message[:110])
        results[logical_id] = messages

    return results


validation = validate_resource_properties(template)

clean = True
for logical_id, messages in validation.items():
    status = "ok" if not messages else str(len(messages)) + " error(s)"
    print(f"  {logical_id:<16} {status}")
    for message in messages:
        clean = False
        print("      ", message)

print()
if clean:
    print("Every resource matches its CloudFormation type schema. Safe to deploy -")
    print("subject to anything the service enforces beyond that schema.")
else:
    print("Fix the above before deploying; each would have cost a rollback.")

  LambdaRole       ok
  BookingLambda    ok
  GatewayRole      ok
  Gateway          ok
  BookingTarget    ok

Every resource matches its CloudFormation type schema. Safe to deploy -
subject to anything the service enforces beyond that schema.


### 6b. Deploy it, idempotently

"Idempotent" means running this cell twice leaves the account in the same state as running it once. With a
stack that falls out of three rules:

1. **Stack does not exist** → `create_stack`.
2. **Stack exists and the template changed** → `update_stack`, which applies only the difference.
3. **Stack exists and nothing changed** → `update_stack` refuses with `No updates are to be performed.` —
   and that refusal is *success*, not failure. It is the one error the function deliberately swallows.
4. **Stack exists in `ROLLBACK_COMPLETE`** (a previous create failed) → delete the husk, then create.

Any other error is re-raised untouched. Swallowing broadly would hide a real failure behind a cheerful
"already deployed"; swallowing exactly one known message is what makes the idempotency honest.

`CAPABILITY_IAM` is passed because the template creates IAM roles — CloudFormation refuses to do that without
an explicit acknowledgement. `validate_template` reports this requirement — and that is close to *all* it
reports. It checks template syntax and intrinsic functions, **not** each resource's property schema. A template
can pass validation and still fail at deploy on a wrong property name, which is exactly what happened to this
one's first run. Only the deploy validates properties.

A fourth rule was added after that failure: a stack whose creation failed is left in `ROLLBACK_COMPLETE`. It
exists, so rule 2 would try to update it — and `update_stack` refuses a stack in that state. The only recovery
is `delete_stack` then create, which `deploy_stack` now does automatically, printing that it is doing so.

The waiter polls every 30 seconds until the stack reaches a terminal state. On failure it raises, and the cell
prints the first failed resource's reason — which is where anything the account forbids will surface.

**What to expect, first run:** `CREATE_COMPLETE` after roughly 1–3 minutes. **Later runs:** `no changes` in
seconds.

**Creates:** two IAM roles, one Lambda function, one Gateway, one Gateway target. Cost: Gateway bills $5 per
million tool invocations and nothing for existing; Lambda is within free tier.

In [21]:
# =====================================================
# 6b. IDEMPOTENT DEPLOY - create, update, or "nothing to do"
# =====================================================
from botocore.exceptions import ClientError, WaiterError

cfn = boto3.client("cloudformation", region_name=REGION)

NO_CHANGES_MESSAGE = "No updates are to be performed"


def stack_status(stack_name):
    """Return a stack's status, or None if it does not exist.

    Args:
        stack_name: The stack name to look up.

    Returns:
        str | None: e.g. "CREATE_COMPLETE", "ROLLBACK_COMPLETE", or None when
            CloudFormation says the stack does not exist. Other errors propagate.
    """
    try:
        return cfn.describe_stacks(StackName=stack_name)["Stacks"][0]["StackStatus"]
    except ClientError as error:
        if "does not exist" in str(error):
            return None
        raise


def first_failure_reason(stack_name):
    """Return the first failed resource and its reason from the stack's events.

    Args:
        stack_name: The stack whose events to scan.

    Returns:
        str: "<LogicalId>: <reason>" for the earliest *_FAILED event, or a
            message saying none was found.
    """
    events = cfn.describe_stack_events(StackName=stack_name)["StackEvents"]
    for event in reversed(events):                 # oldest first
        if event["ResourceStatus"].endswith("_FAILED"):
            return event["LogicalResourceId"] + ": " + event.get("ResourceStatusReason", "(no reason)")
    return "no *_FAILED event found"


def deploy_stack(stack_name, template_body):
    """Create or update a CloudFormation stack and wait for it to settle.

    Idempotent: a first run creates; later runs update; an update with no
    changes is treated as success; a stack left in ROLLBACK_COMPLETE by a
    failed create is deleted and recreated. Exactly one error message is
    swallowed; everything else propagates.

    Args:
        stack_name: Name of the stack. This is the idempotency key.
        template_body: The template as a JSON string.

    Returns:
        str: One of "created", "updated", or "no changes".
    """
    status = stack_status(stack_name)

    # Rule 4, learned from a failed first deploy: a stack whose creation failed
    # sits in ROLLBACK_COMPLETE. It exists, but update_stack refuses it - the
    # only way forward is to delete the husk and create again.
    if status == "ROLLBACK_COMPLETE":
        print("  previous create failed (ROLLBACK_COMPLETE) - deleting the husk first")
        cfn.delete_stack(StackName=stack_name)
        cfn.get_waiter("stack_delete_complete").wait(StackName=stack_name)
        status = None

    if status is not None:
        try:
            cfn.update_stack(
                StackName=stack_name,
                TemplateBody=template_body,
                Capabilities=["CAPABILITY_IAM"],
            )
        except ClientError as error:
            if NO_CHANGES_MESSAGE in str(error):
                return "no changes"
            raise
        waiter_name = "stack_update_complete"
        outcome = "updated"
    else:
        cfn.create_stack(
            StackName=stack_name,
            TemplateBody=template_body,
            Capabilities=["CAPABILITY_IAM"],
        )
        waiter_name = "stack_create_complete"
        outcome = "created"

    try:
        cfn.get_waiter(waiter_name).wait(StackName=stack_name)
    except WaiterError:
        print("stack did not reach a complete state. first failure:")
        print("  ", first_failure_reason(stack_name))
        raise

    return outcome


print("deploying", STACK_NAME, "...")
result = deploy_stack(STACK_NAME, template_body)
status = cfn.describe_stacks(StackName=STACK_NAME)["Stacks"][0]["StackStatus"]

print("result :", result)
print("status :", status)

deploying wanderbot-gateway-demo ...
result : no changes
status : CREATE_COMPLETE


### 6c. Read the endpoint back from the stack

The course copies the Gateway URL out of the console and pastes it into `demo.py`. Here the stack *tells* the
notebook: `GatewayUrl` was declared as an output, so `describe_stacks` returns it.

That removes the paste, but it also changes what the value *is*. It is now derived from the deployment every
time the cell runs — rebuild the stack and the variable follows, with nothing stale left in a source file.

The URL is printed masked. An unauthenticated Gateway URL is a capability — anyone who has it can call the
tools — and notebook output is committed.

**What to expect:** a URL ending in `/mcp` with the host's unique label masked, and the Lambda ARN with its
account masked.

Read-only.

In [22]:
# =====================================================
# 6c. THE GATEWAY URL, FROM STACK OUTPUTS - NOT FROM A PASTE
# =====================================================
# Own client, so this cell works after a kernel restart without re-running 6b.
cfn = boto3.client("cloudformation", region_name=REGION)


def stack_outputs(stack_name):
    """Return a CloudFormation stack's outputs as a plain dict.

    Args:
        stack_name: The stack to read.

    Returns:
        dict: {OutputKey: OutputValue} for every declared output.
    """
    stack = cfn.describe_stacks(StackName=stack_name)["Stacks"][0]
    return {o["OutputKey"]: o["OutputValue"] for o in stack.get("Outputs", [])}


def mask_url(url):
    """Mask the unique host label of a Gateway URL, keeping scheme, domain and path.

    Gateway URLs look like https://<id>.gateway.bedrock-agentcore.<region>.amazonaws.com/mcp.
    The <id> is the capability; the rest is structure worth reading.

    Args:
        url: The full Gateway URL.

    Returns:
        str: The URL with its first host label replaced by asterisks.
    """
    scheme, sep, rest = url.partition("://")
    host, slash, path = rest.partition("/")
    first_label, dot, remainder = host.partition(".")
    return scheme + sep + "*" * len(first_label) + dot + remainder + slash + path


outputs = stack_outputs(STACK_NAME)
GATEWAY_ENDPOINT = outputs["GatewayUrl"]
LAMBDA_ARN = outputs["LambdaArn"]

print("GatewayUrl :", mask_url(GATEWAY_ENDPOINT))
print("LambdaArn  :", mask_arn(LAMBDA_ARN))
print()
print("GATEWAY_ENDPOINT is now derived from the deployment - nothing was pasted.")

GatewayUrl : https://****************************.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp
LambdaArn  : arn:aws:lambda:us-east-1:********6465:function:wanderbot-booking-tools

GATEWAY_ENDPOINT is now derived from the deployment - nothing was pasted.


### 6d. The fallback — the same five resources, built by hand

This cell is **not run by default**: `RUN_FALLBACK = False`. It exists because the stack in 6b is not available
in every account. The first lab account this notebook ran in carried a cost-control policy that intermittently
denied CloudFormation, and the backend was built there through the individual APIs instead — `iam`, `lambda`,
`bedrock-agentcore-control`. That code is kept, verified, as the thing to reach for when 6b is refused.

Without a stack, "idempotent" has to be built per resource, and the pattern is always the same three lines:
*look it up by name; if found, reuse it; otherwise create it.* Five resources, five applications of that
pattern, in dependency order. Running it a second time prints `reused` five times and changes nothing.

Doing it by hand also exposed two things the stack hides, both worth knowing:

- **`create_gateway` does not validate `roleArn`.** It accepted a role that did not exist and returned a
  `READY` Gateway. The check happens later, at `create_gateway_target`, which refuses with *"Gateway execution
  role lacks permission to invoke Lambda function"* if the role cannot reach the function.
- **IAM is eventually consistent, and services see it late.** The invoke permission was attached and confirmed
  present, yet target creation was refused seconds afterwards with exactly that message. The code retries on
  that one string only. CloudFormation handles this ordering for you; by hand, you handle it.

Set `RUN_FALLBACK = True` only if 6b failed with an access error. If 6b succeeded, running this would create a
*second* Gateway with different names alongside the stack's — which is exactly the duplication the stack
exists to prevent.

**Creates (only when enabled):** the same two roles, Lambda, Gateway and target, under `-manual` names.

In [23]:
# =====================================================
# 6d. FALLBACK - HAND-BUILT GET-OR-CREATE, DEFINED BUT NOT RUN
# =====================================================
import io
import time
import zipfile

RUN_FALLBACK = False     # set True only if 6b was denied

# Distinct names so this can never collide with the stack's resources.
FB_LAMBDA_ROLE = "wanderbot-booking-lambda-role-manual"
FB_GATEWAY_ROLE = "wanderbot-gateway-role-manual"
FB_LAMBDA_NAME = "wanderbot-booking-tools-manual"
FB_GATEWAY_NAME = "wanderbot-gateway-manual"
FB_TARGET_NAME = "booking-target"


def get_or_create_role(iam, role_name, service, managed_policy_arns=()):
    """Return an IAM role's ARN, creating the role if absent.

    The idempotency pattern used throughout this cell: look up by name, reuse
    if present, create otherwise. Only NoSuchEntity means "absent".

    Args:
        iam: A boto3 IAM client.
        role_name: The role name - the idempotency key.
        service: The service principal allowed to assume the role.
        managed_policy_arns: Managed policies to attach on creation.

    Returns:
        tuple[str, str]: (role_arn, "reused" or "created").
    """
    try:
        return iam.get_role(RoleName=role_name)["Role"]["Arn"], "reused"
    except ClientError as error:
        if error.response["Error"]["Code"] != "NoSuchEntity":
            raise
    trust = {"Version": "2012-10-17", "Statement": [{
        "Effect": "Allow", "Principal": {"Service": service}, "Action": "sts:AssumeRole"}]}
    arn = iam.create_role(RoleName=role_name, AssumeRolePolicyDocument=json.dumps(trust))["Role"]["Arn"]
    for policy_arn in managed_policy_arns:
        iam.attach_role_policy(RoleName=role_name, PolicyArn=policy_arn)
    return arn, "created"


def get_or_create_lambda(lam, function_name, role_arn, source_text):
    """Return a Lambda's ARN, creating it from inline source if absent.

    Retries on the one error a just-created role produces: "cannot be assumed".

    Args:
        lam: A boto3 Lambda client.
        function_name: The function name - the idempotency key.
        role_arn: The execution role.
        source_text: booking_lambda.py as text; zipped in memory as index.py.

    Returns:
        tuple[str, str]: (function_arn, "reused" or "created").
    """
    try:
        return lam.get_function(FunctionName=function_name)["Configuration"]["FunctionArn"], "reused"
    except ClientError as error:
        if error.response["Error"]["Code"] != "ResourceNotFoundException":
            raise
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.writestr("index.py", source_text)
    for attempt in range(6):
        try:
            fn = lam.create_function(
                FunctionName=function_name, Runtime=LAMBDA_RUNTIME, Role=role_arn,
                Handler="index.lambda_handler", Code={"ZipFile": buffer.getvalue()}, Timeout=10)
            return fn["FunctionArn"], "created"
        except ClientError as error:
            if "cannot be assumed" in str(error) and attempt < 5:
                time.sleep(5)
                continue
            raise


def get_or_create_gateway(agentcore, name, role_arn):
    """Return a Gateway's id, creating it if absent. list_gateways has no name
    filter, so the lookup scans. NOTE: roleArn is NOT validated by the service.

    Args:
        agentcore: A boto3 bedrock-agentcore-control client.
        name: The Gateway name - the idempotency key.
        role_arn: The role the Gateway assumes to reach targets.

    Returns:
        tuple[str, str]: (gateway_id, "reused" or "created").
    """
    for item in agentcore.list_gateways(maxResults=50)["items"]:
        if item["name"] == name:
            return item["gatewayId"], "reused"
    created = agentcore.create_gateway(
        name=name, roleArn=role_arn, protocolType="MCP", authorizerType="NONE")
    return created["gatewayId"], "created"


def wait_for_gateway_ready(agentcore, gateway_id, timeout_seconds=180):
    """Poll get_gateway until READY; raise on FAILED or timeout."""
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        gateway = agentcore.get_gateway(gatewayIdentifier=gateway_id)
        if gateway["status"] == "READY":
            return
        if gateway["status"] == "FAILED":
            raise RuntimeError("Gateway FAILED: " + "; ".join(gateway.get("statusReasons", [])))
        time.sleep(5)
    raise TimeoutError("Gateway not READY within " + str(timeout_seconds) + "s")


def get_or_create_lambda_target(agentcore, gateway_id, name, lambda_arn, tool_definitions):
    """Return a Lambda target's id, creating it if absent. Retries on the
    IAM-propagation refusal "lacks permission to invoke".

    Args:
        agentcore: A boto3 bedrock-agentcore-control client.
        gateway_id: The Gateway to attach to.
        name: The target name - the idempotency key and the ___ prefix.
        lambda_arn: The function the Gateway will invoke.
        tool_definitions: The parsed schema file.

    Returns:
        tuple[str, str]: (target_id, "reused" or "created").
    """
    for item in agentcore.list_gateway_targets(gatewayIdentifier=gateway_id, maxResults=50)["items"]:
        if item["name"] == name:
            return item["targetId"], "reused"
    for attempt in range(8):
        try:
            created = agentcore.create_gateway_target(
                gatewayIdentifier=gateway_id, name=name,
                targetConfiguration={"mcp": {"lambda": {
                    "lambdaArn": lambda_arn, "toolSchema": {"inlinePayload": tool_definitions}}}},
                credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}])
            return created["targetId"], "created"
        except ClientError as error:
            if "lacks permission to invoke" in str(error) and attempt < 7:
                time.sleep(10)
                continue
            raise


if RUN_FALLBACK:
    iam = boto3.client("iam")
    lam = boto3.client("lambda", region_name=REGION)
    agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)

    lambda_role_arn, how1 = get_or_create_role(iam, FB_LAMBDA_ROLE, "lambda.amazonaws.com",
        ["arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"])
    gateway_role_arn, how2 = get_or_create_role(iam, FB_GATEWAY_ROLE, "bedrock-agentcore.amazonaws.com")
    if "created" in (how1, how2):
        time.sleep(10)                     # IAM propagation
    fb_lambda_arn, how3 = get_or_create_lambda(lam, FB_LAMBDA_NAME, lambda_role_arn, lambda_source)
    iam.put_role_policy(RoleName=FB_GATEWAY_ROLE, PolicyName="InvokeBookingLambda",
        PolicyDocument=json.dumps({"Version": "2012-10-17", "Statement": [{
            "Effect": "Allow", "Action": "lambda:InvokeFunction", "Resource": fb_lambda_arn}]}))
    fb_gateway_id, how4 = get_or_create_gateway(agentcore, FB_GATEWAY_NAME, gateway_role_arn)
    wait_for_gateway_ready(agentcore, fb_gateway_id)
    fb_target_id, how5 = get_or_create_lambda_target(agentcore, fb_gateway_id, FB_TARGET_NAME, fb_lambda_arn, tool_schema)

    GATEWAY_ENDPOINT = agentcore.get_gateway(gatewayIdentifier=fb_gateway_id)["gatewayUrl"]
    LAMBDA_ARN = fb_lambda_arn
    for label, how in [("LambdaRole", how1), ("GatewayRole", how2), ("Lambda", how3), ("Gateway", how4), ("Target", how5)]:
        print(f"  {label:<12} {how}")
    print("GATEWAY_ENDPOINT :", mask_url(GATEWAY_ENDPOINT))
else:
    print("Fallback defined, not run. 6b succeeded, so the stack's Gateway is in use.")
    print("Set RUN_FALLBACK = True only if 6b was denied in your account.")

Fallback defined, not run. 6b succeeded, so the stack's Gateway is in use.
Set RUN_FALLBACK = True only if 6b was denied in your account.


## Step 7 — The real handshake, and what comes back

Step 5 built the MCP client against a placeholder and proved construction is lazy. Now the endpoint is real, so
`with client:` does what Part A could only describe: open the transport, complete the MCP `initialize`
handshake, and hand back a live session.

### 7a. Discover the tools

`list_tools_sync()` asks the Gateway's MCP server for every registered tool. Three predictions, written before
running:

1. **Two tools**, because the target was registered with a two-entry schema.
2. **Prefixed names.** The Gateway exposes each tool as `<target>___<tool>` — so `booking-target___get_booking`,
   not `get_booking`. That prefix is exactly what Step 3b's handler splits on `___` to recover the function name.
   The convention runs end to end: the Gateway adds it going out, the Lambda strips it coming in.
3. **The schema round-trips unchanged.** The description and input schema that come back should be
   byte-for-byte what `booking_lambda.json` says — despite having been re-cased to PascalCase for CloudFormation
   in 6a, registered on the target, and now returned over MCP as lowercase JSON Schema. Same contract, three
   spellings, zero drift.

A fourth thing to look at: `tool_spec["inputSchema"]` arrives nested under a `json` key. Step 2's `@tool` produced
the identical nesting. That is `AgentTool` normalising a local function and a remote MCP tool into one shape,
which is the concrete reason Step 5b's "same interface" claim holds.

**Cost:** two Gateway operations (initialize, list). Under a thousandth of a cent.

In [ ]:
# =====================================================
# 7a. LIVE DISCOVERY - list_tools_sync AGAINST THE REAL GATEWAY
# =====================================================
import time

live_client = MCPClient(lambda: streamable_http_client(url=GATEWAY_ENDPOINT))

started = time.time()
with live_client:
    handshake_seconds = time.time() - started

    gateway_tools = live_client.list_tools_sync()
    discovery_seconds = time.time() - started

    print(f"handshake complete in {handshake_seconds:.2f}s; tools listed by {discovery_seconds:.2f}s")
    print("tools discovered:", len(gateway_tools))
    print()

    for tool in gateway_tools:
        spec = tool.tool_spec
        # The Gateway prefixes every tool with its target name and ___ .
        target_part, separator, bare_name = tool.tool_name.partition("___")
        print("  tool_name   :", tool.tool_name)
        print("    target    :", target_part, " | bare tool:", bare_name)
        print("    description:", spec["description"][:72])
        # Same 'json' nesting that Step 2's local @tool produced.
        print("    inputSchema keys:", list(spec["inputSchema"].keys()))
        print("    parameters:", list(spec["inputSchema"]["json"]["properties"]))
        print()

# Prove the round trip: compare what came back with what we registered in Step 4.
registered = {t["name"]: t for t in tool_schema}
for tool in gateway_tools:
    bare = tool.tool_name.partition("___")[2]
    came_back = tool.tool_spec["inputSchema"]["json"]
    went_in = registered[bare]["inputSchema"]
    print(f"{bare:<24} schema identical to booking_lambda.json: {came_back == went_in}")

### 7b. One direct tool call — the proof that was missing

Discovery only touches the Gateway's registry. It says nothing about whether the Gateway can actually reach the
Lambda — and that is the one claim nothing has verified yet. 6e noted that `create_gateway` never checks the
role it is given, and CloudFormation did not either. A wrong `GatewayRole` would have produced a `READY` Gateway
that lists its tools perfectly and fails on first use.

So before any model is involved, call one tool directly with `call_tool_sync`. Same session, no agent, known
input: `booking_ref = "BK-1001"`. If Alice Johnson comes back, the whole chain is proven — MCP to Gateway,
Gateway assumes the role, role invokes the Lambda, handler strips the prefix, function runs.

**Look closely at the shape of what comes back.** The Lambda returns `{"statusCode": 200, "body": "..."}` where
`body` is a JSON *string*, and inside it `result` is *another* JSON string holding the booking. That is three
levels of JSON-inside-a-string, and the model will receive the whole thing as one escaped text block. It happens
because the handler returns an HTTP-style envelope — `statusCode` and a stringified `body` — to a caller that is
not HTTP. The README's own talking point is "no HTTP routing", yet the handler still wraps like an API Gateway
response. The model copes, which is why a working demo never surfaces it; a cleaner handler would return the
result dict directly.

**What to expect:** `status: success`, and Alice Johnson's BK-1001 recoverable after three `json.loads`.

**Cost:** one Gateway tool invocation, one Lambda invocation.

In [ ]:
# =====================================================
# 7b. ONE DIRECT TOOL CALL - NO MODEL, KNOWN INPUT
# =====================================================


def unwrap_lambda_result(mcp_result):
    """Peel the Lambda's HTTP-style envelope off an MCP tool result.

    The handler returns {"statusCode": 200, "body": "<json string>"} and the
    body's "result" field is itself a JSON string - three layers. This undoes
    them so the booking is a plain dict.

    Args:
        mcp_result: The MCPToolResult returned by call_tool_sync.

    Returns:
        dict: The booking (or the handler's plain-text message, if the inner
            result was not JSON).
    """
    outer_text = mcp_result["content"][0]["text"]       # layer 1: MCP text block
    envelope = json.loads(outer_text)                   # {"statusCode": .., "body": "..."}
    body = json.loads(envelope["body"])                 # layer 2: {"result": "..."}
    try:
        return json.loads(body["result"])               # layer 3: the booking itself
    except json.JSONDecodeError:
        return body["result"]                           # e.g. "No booking found ..."


with live_client:
    result = live_client.call_tool_sync(
        tool_use_id="direct-probe-1",
        name="booking-target___get_booking",
        arguments={"booking_ref": "BK-1001"},
    )

print("status      :", result["status"])
print()
print("raw text the model would see (first 140 chars):")
print("  ", result["content"][0]["text"][:140], "...")
print()

booking = unwrap_lambda_result(result)
print("after unwrapping three layers of JSON:")
for key in ("booking_ref", "customer", "flight", "route", "fare_usd", "status"):
    print(f"  {key:<12} {booking[key]}")
print()
print("Alice came back. The Gateway assumed GatewayRole and invoked the Lambda -")
print("the one link that neither create_gateway nor CloudFormation ever checked.")

### What the output proved

**The handshake is sub-second and discovery is two round trips.** Construction was free (Step 5a); `with
client:` cost about half a second to initialise, and `list_tools_sync()` a further half. This is the same
`initialize` → `tools/list` sequence the course notes show with `curl`, done by the SDK.

**The `___` convention runs end to end.** `booking-target___get_booking` out of the Gateway;
`split("___", 1)[-1]` in the handler. Neither side knows the other's code — they agree on a separator.

**The contract survived three spellings without drift.** `schema identical to booking_lambda.json: True` for
both tools. Lowercase in the file, PascalCase into CloudFormation, lowercase back out over MCP. Step 4 called the
schema "the bridge"; this is the bridge load-tested.

**And the role works.** A direct `call_tool_sync` returned Alice's booking, which means the Gateway assumed
`GatewayRole` and that role invoked the Lambda. Nothing in Step 6 verified this — `create_gateway` accepts any
ARN, and CloudFormation trusted it too. The first *invocation* is the only test, which is why this step makes
one before letting a model near it.

**The thing to fix if this were production:** the triple-wrapped JSON. The handler's `statusCode`/`body` envelope
is API Gateway muscle memory; behind an AgentCore Gateway it just makes the model parse escaped strings.

With discovery and invocation both proven, the agent itself adds nothing new to the transport — it only decides
*which* tool to call. Step 8 lets it.

## Step 8 — Let the model choose the tool

Everything below the model is now proven: the Gateway lists its tools (7a) and invokes the Lambda when asked
(7b). The agent adds exactly one thing to that — it reads the user's sentence and *decides which tool to call
with which arguments*. The transport does not change.

`invoke_through_gateway` from Step 5c is used **unchanged**. It was defined against a placeholder endpoint four
steps ago; the only difference now is that `GATEWAY_ENDPOINT` is real. If the function needed editing to work
against a live Gateway, Step 5 would have been teaching the wrong shape.

Two questions, both from the README. The first names a booking reference, which should map to `get_booking`.
The second names an email, which should map to `list_bookings_by_email` and return *two* bookings — Alice has
BK-1001 and BK-1002. A third question is deliberately unanswerable: a reference that does not exist. The
handler returns a plain "No booking found" string for that, and the model should relay it rather than invent a
booking — the system prompt says so, and this is the test of whether it listens.

**What to expect:** three answers, each streamed live as the model produces them, each built from real tool
output. The `Tool #1: booking-target___...` line Strands prints before each answer is the model's choice made
visible — the prefixed name, straight from discovery. The model never learns the tool was remote; this looks
identical to a local `@tool` run.

**Cost:** three model calls to nova-2-lite, three Gateway invocations, three Lambda invocations. A few cents.

In [ ]:
# =====================================================
# 8. THE AGENT, THROUGH THE GATEWAY - invoke_through_gateway UNCHANGED FROM 5c
# =====================================================
from strands.models import BedrockModel

model = BedrockModel(model_id="us.amazon.nova-2-lite-v1:0", region_name=REGION)

QUESTIONS = [
    "Can you look up booking BK-1001?",
    "What bookings does alice@example.com have?",
    "Please look up booking BK-9999.",          # does not exist - should not be invented
]


# invoke_through_gateway builds a fresh Agent per call and keeps discovery and
# invocation inside one MCP session - exactly as defined in 5c, untouched.
for number, question in enumerate(QUESTIONS, start=1):
    print(f"=== Q{number}: {question}")
    # The answer streams to the output above as the model produces it - that is
    # Strands' default callback handler, not a print here. Only metrics follow.
    response = invoke_through_gateway(GATEWAY_ENDPOINT, question, model)
    print()
    usage = response.metrics.accumulated_usage
    print("   tokens :", usage["inputTokens"], "in /", usage["outputTokens"], "out")
    print("   stop   :", response.stop_reason)
    print()

### 8b. The other end of the wire — the Lambda's own log

The model's answer is one end of the chain. The Lambda's CloudWatch log is the other, and it is the only
evidence that does not pass through the model. Step 3b's handler prints `Tool called: <name> | Event: <args>` on
every invocation, and those lines are in `/aws/lambda/wanderbot-booking-tools`.

Read them back and three things are visible that nothing else shows: the exact tool name the handler resolved
after stripping `___`, the *arguments the model chose* arriving as the plain event the handler expected, and
one line per invocation — so the count should match the number of tool calls made in Steps 7 and 8.

This is the terminal the course transcript switches to mid-demo to show "the agent has reached out to this
Lambda function". Same evidence, read programmatically.

**What to expect:** one `Tool called` line per invocation so far, with `get_booking` for the reference lookups
and `list_bookings_by_email` for the email query. The `BK-9999` call appears too — the handler ran; it just
found nothing.

Read-only — CloudWatch Logs query. Log delivery lags a few seconds, so a very recent call may not show yet.

In [ ]:
# =====================================================
# 8b. THE LAMBDA'S OWN LOG - EVIDENCE THAT DOES NOT PASS THROUGH THE MODEL
# =====================================================
import time

logs = boto3.client("logs", region_name=REGION)
LOG_GROUP = "/aws/lambda/" + LAMBDA_FUNCTION_NAME


def recent_tool_calls(log_group, minutes=30):
    """Return the handler's own 'Tool called' lines from the last N minutes.

    Args:
        log_group: The Lambda's CloudWatch log group name.
        minutes: How far back to look.

    Returns:
        list[str]: One line per invocation, oldest first.
    """
    since_ms = int((time.time() - minutes * 60) * 1000)
    found = []
    token = None
    while True:
        kwargs = {"logGroupName": log_group, "filterPattern": '"Tool called"', "startTime": since_ms}
        if token:
            kwargs["nextToken"] = token
        page = logs.filter_log_events(**kwargs)
        found.extend(event["message"].strip() for event in page["events"])
        token = page.get("nextToken")
        if not token:
            return found


time.sleep(5)              # CloudWatch delivery lags the invocation slightly
calls = recent_tool_calls(LOG_GROUP)

print("invocations logged by the handler in the last 30 minutes:", len(calls))
for line in calls:
    print("  ", line[:96])
print()
print("Each line is the handler printing what the Gateway handed it: the resolved tool")
print("name, and the model's chosen arguments as the plain event booking_lambda expects.")

### What the output proved

**The agent code did not change.** `invoke_through_gateway`, written in Step 5c against `example.invalid`, ran
against the live Gateway as-is. The README's talking point — "the agent code barely changed; we swapped `@tool`
for `MCPClient`" — is now a demonstrated fact rather than a slogan.

**The model routed correctly on all three.** A reference went to `get_booking`, an email to
`list_bookings_by_email`, and the nonexistent `BK-9999` produced the handler's "No booking found" relayed
honestly rather than a fabricated booking. The model made those choices from the *descriptions* in
`booking_lambda.json` — the schema was not only the registration contract but the model's instructions.

**The message shape is the lab 4 shape.** Four messages for a tool-using turn, `text → toolUse → toolResult →
text`. Nothing in the dump says "remote". Below the `AgentTool` boundary the model is blind to location, exactly
as Step 5b argued from the type hierarchy.

**And the Lambda saw the model's arguments.** `Tool called: get_booking | Event: {"booking_ref": "BK-1001"}` in
CloudWatch is the model's tool-use request after the Gateway unwrapped it — the end-to-end path observed from
the far end, independent of anything the model said.

**The cost that did not show:** the triple-wrapped JSON from 7b. nova-2-lite parsed it without complaint, which
is precisely why the handler's HTTP envelope survives in demos. It is still ~2,200 input tokens for a one-line
question, and most of that is the two tool schemas and the escaped result being re-sent — the same "being
reminded has a bill" lesson from lab 4, now with a remote tool's result in the bill.

## Step 9 — Tear it down, and verify independently

A Gateway with `AuthorizerType: NONE` is a public, unauthenticated endpoint that can invoke a Lambda in this
account. It costs nothing while idle — but it is a capability sitting in the open, and the lab account is
shared. It should not outlive the lesson.

This is where the CloudFormation decision pays off one last time. The hand-built fallback in 6d would need five
deletes in reverse dependency order — target, Gateway, Lambda, then the roles after detaching their policies.
The stack needs **one call**. `delete_stack` removes all five, in the right order, and the
`stack_delete_complete` waiter blocks until CloudFormation confirms the stack is gone.

Then the habit from the memory lab: **verify independently of the call that claimed success.** `describe_stacks`
should fail with "does not exist", and — because a stack delete can leave orphans if a resource's deletion is
blocked — the cell also checks the account directly: no Gateway named `wanderbot-gateway`, no Lambda named
`wanderbot-booking-tools`.

**What to expect:** `DELETE_COMPLETE` (or the stack simply absent) in about a minute, then three independent
"gone" confirmations.

**Destroys:** the stack and all five resources. Irreversible, and nothing in it is precious — Step 6 recreates
it in under a minute.

In [ ]:
# =====================================================
# 9. ONE CALL TO REMOVE FIVE RESOURCES, THEN VERIFY INDEPENDENTLY
# =====================================================
from botocore.exceptions import ClientError

cfn = boto3.client("cloudformation", region_name=REGION)
agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)
lam = boto3.client("lambda", region_name=REGION)


def delete_stack_and_wait(stack_name):
    """Delete a CloudFormation stack and block until CloudFormation confirms it is gone.

    Idempotent: a stack that does not exist is reported, not treated as an error.

    Args:
        stack_name: The stack to delete.

    Returns:
        str: "deleted" or "already absent".
    """
    try:
        cfn.describe_stacks(StackName=stack_name)
    except ClientError as error:
        if "does not exist" in str(error):
            return "already absent"
        raise
    cfn.delete_stack(StackName=stack_name)
    cfn.get_waiter("stack_delete_complete").wait(StackName=stack_name)
    return "deleted"


print("deleting", STACK_NAME, "...")
print("result:", delete_stack_and_wait(STACK_NAME))
print()

# ---- verify from three independent angles ----
try:
    cfn.describe_stacks(StackName=STACK_NAME)
    print("stack      : STILL PRESENT - investigate")
except ClientError as error:
    print("stack      :", "gone" if "does not exist" in str(error) else str(error)[:60])

gateways_left = [g["name"] for g in agentcore.list_gateways(maxResults=50)["items"] if g["name"] == GATEWAY_NAME]
print("gateway    :", "gone" if not gateways_left else "STILL PRESENT - " + str(gateways_left))

try:
    lam.get_function(FunctionName=LAMBDA_FUNCTION_NAME)
    print("lambda     : STILL PRESENT - investigate")
except ClientError as error:
    print("lambda     :", "gone" if error.response["Error"]["Code"] == "ResourceNotFoundException" else error.response["Error"]["Code"])

print()
print("Nothing left running. The unauthenticated endpoint no longer exists.")

## What this notebook established

| Step | The question | The answer |
| --- | --- | --- |
| 1 | will this run safely? | credentials from the environment, identifiers masked in every output |
| 2 | why move tools out? | a `@tool` couples logic, runtime and permissions to the agent's deployment |
| 3 | what is behind the Gateway? | plain functions, plus a handler that routes on a name the Gateway supplies |
| 4 | what does the schema do? | the same `name`/`description`/`inputSchema` contract `@tool` generated — authored by hand |
| 5 | how does the agent consume remote tools? | an MCP session; `MCPAgentTool` and `DecoratedFunctionTool` share `AgentTool` |
| 6 | how is the backend built? | one CloudFormation stack, idempotent, endpoint as an output — after four tries |
| 7 | does the Gateway really work? | discovery returns `target___tool` names; one direct call reaches the Lambda |
| 8 | does the agent need to change? | no — Step 5c's function ran unmodified against the live endpoint |
| 9 | what is left behind? | nothing, in one call, verified from three angles |

**The one-sentence version.** A tool is a contract — a name, a description, an input schema — and the agent
consumes that contract the same way whether the code behind it runs in-process or behind a Gateway in a Lambda;
the Gateway's job is to hold the contract, discover it over MCP, and translate each call.

**What the failures taught, which a clean run would not have.** Four deploys to get Step 6 right, and each
failure was a real fact about the system: the schema file's casing is for one API and CloudFormation wants
another, recursively; `validate_template` validates almost nothing about properties; the Gateway service
enforces a field its CloudFormation type marks optional; and a failed stack leaves a `ROLLBACK_COMPLETE` husk
that cannot be updated. The offline property validator in 6a′ exists because of those four, and would have
prevented two of them.

**Deliberately not covered.** `@app.entrypoint` and `app.run()` were never exercised — everything ran in this
kernel, so `demo.py`'s deployed shape is described but not run. Authentication is absent by design:
`AuthorizerType: NONE` is the course's development setting, and the later module on AgentCore Identity is where
that changes. And the hand-built fallback in 6d was written and verified in a different account but not run in
this one, because the stack succeeded.

**The thing to fix if this were real.** The Lambda's HTTP-style `statusCode`/`body` envelope triple-wraps every
result in escaped JSON. The model copes; a production handler would return the dict.